# N051 · 栈、括号与路径归约

**目标：** 把最近未匹配对象作为栈顶状态。

**先修：** N008, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** LIFO；括号匹配；消除相邻项；目录栈；空栈。

**配套讲解来源：** [同名逐章意见](../../comment/051_stack_matching_paths.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章讲一个统一的小思想：**"最新的还没完成的事"永远排在最前面等着被处理**。看起来三件不相关的事，其实都是这一句话：

1. **括号匹配**：给一个只含括号的字符串，判断它是否合法。比如 `()[]{}` 是合法的；`([)]` 不合法——`)` 想闭合的是最近的 `(`，但中间隔了一个没关的 `[`。
2. **简化 Unix 路径**：把 `/a/./b/../../c/` 这种含 `.`（当前目录）和 `..`（上级目录）的路径变成最简形式。输入 `/a/./b/../../c/` → 输出 `/c`。为什么：先进 a，再进 b；第一个 `..` 退掉 b，第二个 `..` 退掉 a；最后进 c，所以只剩 `/c`。
3. **删除相邻重复项**：反复删除字符串里相邻的两个相同字符，直到删不动为止。输入 `abbaca` → 输出 `ca`。为什么：中间的 `bb` 先消掉变成 `aaca`，新的 `aa` 又相邻了，再消变成 `ca`，没有新的相邻对，结束。

三者的共同点（“思路推导”部分 原话）：右括号只能匹配**最近的尚未闭合**的左括号；`..` 只能取消**最近一级**目录；相邻删除一次只能消掉**栈顶**那个重复。这就是栈"后进先出"的用武之地。

## 2. 基础知识：先读懂这些词

- **栈（stack）**：只允许在一头进、同一头出的容器，进和出都在"栈顶"。就像一摞盘子，最后放上去的最先被拿走，所以叫后进先出（LIFO：Last In, First Out）。
- **LIFO（后进先出）**：栈的核心性质。本章所有算法都靠它保证"最新入栈的对象最先被后续输入处理"。
- **归约（reduction）**：把输入中已经能确定结果的一小段就地消掉或合并掉。比如 `(` 和 `)` 凑成一对就"归约"完成，从栈里消失。
- **括号匹配**：每种左括号必须由同种右括号按嵌套顺序闭合。栈里存的是"还没等到自己右括号的左括号"。
- **消除相邻项**：新来的字符如果和栈顶相同，两个一起消失；这个消失还可能让更早的字符变成新的相邻对，所以要用栈一层层往外消。
- **目录栈**：把路径按 `/` 切成一段一段，普通目录名进栈，`..` 出栈，最后栈里从底到顶就是最简路径。
- **空栈**：栈里一个元素都没有。它有两种含义要分清：括号题里"读右括号时栈空"说明多了个右括号（非法）；路径题里"遇到 `..` 时栈空"说明已经在根目录，忽略即可（不算错误）。
- **输入契约**：“思路推导”部分 特意强调三题的字符规则不能混用。比如路径里 `...` 是个普通目录名（三个点的文件名），不是两个 `..`；括号题里出现非括号字符直接判非法。

## 3. 思路推导：从小例子开始

**is_valid_parentheses 的步骤：**

1. 从左到右扫描，遇到左括号 `([{` 就压栈——它"尚未完成"，必须等着。
2. 遇到右括号，就去栈顶找它的搭档：栈空（没有等待的左括号）或者弹出来的左括号类型不对，都直接判 False。
3. 遇到其他字符也判 False（本章契约只允许括号字符）。
4. 扫完后栈必须是空的——还剩左括号就说明有人没被闭合。

**simplify_path 的步骤：**

1. 用 `path.split('/')` 把路径切段。连续斜杠会切出空串 `''`，斜杠结尾也会切出一个尾随 `''`。
2. 空串和 `'.'` 表示"原地不动"，直接跳过。
3. `'..'` 表示回上级：栈不空就弹出一项；栈已空（在根目录）就什么都不做。
4. 其他任何串（包括 `'...'` 这种）都是普通目录名，压栈。
5. 最后 `'/'.join(stack)` 拼回路径，前面补 `/`。

**手算演示：`/a/./b/../../c/` 切段后是 `['', 'a', '.', 'b', '..', '..', 'c', '']`——**

| 片段 | 动作 | 栈 |
|------|------|-----|
| `''` | 跳过 | `[]` |
| `'a'` | 压栈 | `['a']` |
| `'.'` | 跳过 | `['a']` |
| `'b'` | 压栈 | `['a','b']` |
| `'..'` | 弹栈 | `['a']` |
| `'..'` | 弹栈 | `[]` |
| `'c'` | 压栈 | `['c']` |
| `''` | 跳过 | `['c']` |

拼回：`'/' + 'c'` = `/c`。

**remove_adjacent_duplicates 的手算演示（“运行示例”部分 的表格，输入 `abbaca`）：**

| 新字符 | 动作 | 归约后前缀（栈） |
|--------|------|------------------|
| a | 栈空，压栈 | a |
| b | 栈顶 a≠b，压栈 | ab |
| b | 栈顶 b=b，弹出 | a |
| a | 栈顶 a=a，弹出 | （空） |
| c | 栈空，压栈 | c |
| a | 栈顶 c≠a，压栈 | ca |

最终栈内容 `ca` 就是答案。注意第 4 步：`bb` 消掉后，前面的 `a` 和新来的 `a` 变成相邻——这就是为什么必须用栈，数组从头扫一遍是发现不了这种"间接相邻"的。

## 4. 核心代码：is_valid_parentheses

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def is_valid_parentheses(s):
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    for c in s:
        if c in '([{':
            stack.append(c)
        elif c in pairs:
            if not stack or stack.pop() != pairs[c]:
                return False
        else:
            return False
    return not stack
```

### 逐段读懂代码

**括号匹配（对应实现）：**

```python
def is_valid_parentheses(s):
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    for c in s:
        if c in '([{':
            stack.append(c)
        elif c in pairs:
            if not stack or stack.pop() != pairs[c]:
                return False
        else:
            return False
    return not stack
```

- `pairs={')':'(',...]}` 把"右括号 → 它期待的左括号"存成字典，右括号是键、左括号是值。
- `if c in '([{': stack.append(c)`：左括号只负责"登记等待"，不需要区分种类。
- `if not stack or stack.pop()!=pairs[c]: return False` 是一行判两件事：栈空说明这个右括号前面没有任何左括号可配（多了个右括号）；`stack.pop()` 弹出最近的左括号，如果和 `pairs[c]` 要求的那种不一样（比如 `[)`），也非法。注意 Python 的 `or` 有短路效果，栈空时不会再去 pop。
- `else: return False`：既不是左括号也不是右括号的字符一律非法，这是本章的输入契约。
- `return not stack`：字符串读完时栈空才返回 True，栈里剩左括号就返回 False。返回值直接用了布尔表达式的紧凑写法。

**路径简化（对应实现）：**

```python
def simplify_path(path):
    if not path.startswith('/'):
        raise ValueError('absolute Unix path required')
    stack = []
    for part in path.split('/'):
        if part in ('', '.'):
            continue
        if part == '..':
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return '/' + '/'.join(stack)
```

- 第一行 `if not path.startswith('/')` 是在挡非法输入：本章只处理以 `/` 开头的绝对路径，相对路径直接抛异常。
- `if part in ('','.'): continue`：空片段（来自连续斜杠或首尾斜杠）和当前目录标记都不影响位置，跳过。
- `if part=='..': if stack: stack.pop()`：回上级时先看栈空不空；在根目录再往上还是根目录，所以栈空时安静地忽略，这是和括号题最大的区别——同样是"空栈时来了一个消除符"，一个判错，一个忽略。
- `return '/'+ '/'.join(stack)`：栈空时 `join` 的结果是空串，正好拼出根目录 `'/'`。

**消除相邻重复（对应实现）：**

```python
def remove_adjacent_duplicates(s):
    stack = []
    for c in s:
        if stack and stack[-1] == c:
            stack.pop()
        else:
            stack.append(c)
    return ''.join(stack)
```

- 整个函数只有四种情况的组合：栈非空且栈顶相同→弹出；否则（栈空或栈顶不同）→压入。
- `if stack and stack[-1]==c` 先判栈非空再取 `stack[-1]`，顺序不能反，否则空栈取 `[-1]` 会直接抛 IndexError。
- 这里不需要 `pairs` 那种配对表，因为"能消除的条件"就是值相等，比括号匹配简单一档。
- `''.join(stack)` 把栈从底到顶连成答案——栈底的字符是最早留下来的，顺序天然正确。

## 5. 分段实现：按顺序运行

**本章接口：** `is_valid_parentheses(s)`、`simplify_path(path)`、`remove_adjacent_duplicates(s)`

### is_valid_parentheses

In [1]:
def is_valid_parentheses(s):
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    for c in s:
        if c in '([{':
            stack.append(c)
        elif c in pairs:
            if not stack or stack.pop() != pairs[c]:
                return False
        else:
            return False
    return not stack

### simplify_path

In [2]:
def simplify_path(path):
    if not path.startswith('/'):
        raise ValueError('absolute Unix path required')
    stack = []
    for part in path.split('/'):
        if part in ('', '.'):
            continue
        if part == '..':
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return '/' + '/'.join(stack)

### remove_adjacent_duplicates

In [3]:
def remove_adjacent_duplicates(s):
    stack = []
    for c in s:
        if stack and stack[-1] == c:
            stack.pop()
        else:
            stack.append(c)
    return ''.join(stack)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='abbaca'; stack=[]; rows=[]
for c in s:
    if stack and stack[-1]==c: stack.pop()
    else: stack.append(c)
    rows.append((c,''.join(stack)))
show_table(['新字符','归约后前缀'],rows)

新字符,归约后前缀
a,a
b,ab
b,a
a,
c,c
a,ca


## 7. 正确性、适用条件与复杂度

栈始终代表已读前缀中不能再直接归约的部分；新的闭合或取消操作只可能影响栈顶。括号最终栈空才表示每个左端都有匹配右端。

**代价：** O(n)时间、O(n)最坏辅助空间。路径处理是字符串规范化，不访问文件系统或符号链接。

### 展开理解

- **为什么对**：“正确性与复杂度”部分 说"栈始终代表已读前缀中不能再直接归约的部分"。翻译过来就是：任何时刻，栈里的东西都是"目前还消不掉/还等着配对"的，已经确定的结果从来不会留在栈里。而每一次新的闭合、`..` 或重复字符，只可能跟**最近**的那个未完成对象发生关系——最近的那个就在栈顶。所以只看栈顶一个元素就够，不需要回头翻整个历史。括号题最后要求栈空，是因为"合法"等价于"每个左括号都被自己的右括号消掉了"，一个都不许剩。
- **复杂度**：三个函数都是把输入扫一遍，每个字符最多压栈一次、弹栈一次，所以时间是 O(n)；栈最坏要把整个字符串存进去（比如 `((((`），辅助空间是 O(n)。n=10⁶ 个字符时也就是百万级的基本操作，一眨眼跑完。
- **前提**：括号题要求输入只含括号字符；路径题要求绝对路径、且只做字符串规范化——“正确性与复杂度”部分 特别声明它**不访问真实文件系统、不处理符号链接**，`..` 只是文本层面的"退一级"。

## 8. 单元测试：每个用例在检查什么

- `assert is_valid_parentheses('()[]{}')`：正常情况——三种括号各自独立闭合。
- `assert not is_valid_parentheses('([)]')`：交叉嵌套。这条专测"右括号必须匹配最近的左括号"：`)` 来的时候栈顶是 `[` 而不是 `(`，立刻露馅。如果实现错了去栈底找，这条会失败。
- `assert is_valid_parentheses('') and not is_valid_parentheses('(')`：两个边界。空串是合法的（没有任何未闭合）；单个左括号读到结尾栈非空，必须 False——这条专防"忘写 `return not stack`"的实现。
- `assert simplify_path('/a/./b/../../c/')=='/c'`：正常情况，混合了 `.`、两次 `..` 和尾随斜杠。
- `assert simplify_path('////../')=='/'`：特殊值——连续四个斜杠切出多个空串，且 `..` 出现在根目录（栈空），必须安静忽略，结果还是根目录 `/`。
- `assert simplify_path('/.../a/..')=='/...'`：陷阱题——`...` 不是两个 `..`，它是一个合法的（奇怪的）目录名。这条专防把 `..` 的判断写成"包含两个点"或用 `startswith` 之类的宽松写法。
- `assert remove_adjacent_duplicates('abbaca')=='ca'`：正常情况，覆盖"消除后产生新的相邻对"的连锁反应（正是 “运行示例”部分 表格演的那条）。
- `assert remove_adjacent_duplicates('aaaa')==''`：偶数个相同字符会两两消光，结果是空串；这也顺带测了栈空时继续入栈/出栈不出错。

In [5]:
assert is_valid_parentheses('()[]{}')

assert not is_valid_parentheses('([)]')

assert is_valid_parentheses('') and (not is_valid_parentheses('('))

assert simplify_path('/a/./b/../../c/') == '/c'

assert simplify_path('////../') == '/'

assert simplify_path('/.../a/..') == '/...'

assert remove_adjacent_duplicates('abbaca') == 'ca'

assert remove_adjacent_duplicates('aaaa') == ''

print('N051: 所有本章断言通过')

N051: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 支持多类括号。

**练习 2：** 解释路径里的...不是上级目录。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（支持多类括号）**：本章代码其实已经支持 `()[]{}` 三类了，所以这道练习可以往两个方向做深：一是把"合法"的契约写清楚（允许嵌套、不允许交叉），自己列 5 个手算例子，其中至少包含 `([)]`（必须 False）和 `{[]()}`（必须 True）；二是试试推广到任意自定义配对表，比如 `<a>...</a>` 这种长标签对——提示：闭符号是字符串而不是单字符时，判断"当前读到的是开还是闭"就不能用 `in '([{'` 一个字符 membership 搞定了，想想怎么按前缀区分。
- **练习 2（解释路径里的 `...` 不是上级目录）**：提示：Unix 路径规范里只有 `.` 和 `..` 是保留名，`...`、`....` 都是普通文件名。练习的重点是把"相等比较 `part=='..'`"和"包含点的模糊匹配"区分开。可以设计边界用例：`/...` → `/...`，`/a/.../..` → `/a`，`/..` → `/`。手算时把每一段写进表格（就像第三节的演示），逐段标注"跳过/压栈/弹栈"，你会看到 `...` 走的是压栈那条分支。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def is_valid_parentheses(s):
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    for c in s:
        if c in '([{':
            stack.append(c)
        elif c in pairs:
            if not stack or stack.pop() != pairs[c]:
                return False
        else:
            return False
    return not stack

def simplify_path(path):
    if not path.startswith('/'):
        raise ValueError('absolute Unix path required')
    stack = []
    for part in path.split('/'):
        if part in ('', '.'):
            continue
        if part == '..':
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return '/' + '/'.join(stack)

def remove_adjacent_duplicates(s):
    stack = []
    for c in s:
        if stack and stack[-1] == c:
            stack.pop()
        else:
            stack.append(c)
    return ''.join(stack)

# 示例与回归测试
assert is_valid_parentheses('()[]{}')

assert not is_valid_parentheses('([)]')

assert is_valid_parentheses('') and (not is_valid_parentheses('('))

assert simplify_path('/a/./b/../../c/') == '/c'

assert simplify_path('////../') == '/'

assert simplify_path('/.../a/..') == '/...'

assert remove_adjacent_duplicates('abbaca') == 'ca'

assert remove_adjacent_duplicates('aaaa') == ''

print('N051: 所有本章断言通过')

N051: 所有本章断言通过


## 11. 代表题与迁移

- **20. Valid Parentheses**：练"左括号压栈、右括号查栈顶配对、结尾栈必须空"三步套路，本章第一个函数就是它。
- **71. Simplify Path**：练目录栈——切段、跳过 `.` 与空段、`..` 弹栈（根目录时忽略）、拼回路径。
- **1047. Remove All Adjacent Duplicates In String**：练"新元素与栈顶相同就一起消失"的相邻消除，以及连锁消除为什么必须用栈。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。